# Skin Disease Classification — Model Training
**Branch:** `feature/model-training`  
**Dataset:** Skin Disease Classification (Mendeley, Jul 2024)  
**Classes:** Acne, Vitiligo, Hyperpigmentation, Nail Psoriasis, SJS-TEN  

### Notebook Structure
1. Setup & Imports
2. Data Loading & Preprocessing
3. Part A — Baseline CNN from Scratch
4. Part B — Transfer Learning with ResNet50
5. Compare Results

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Config (single source of truth — edit here only) ──────────────────────────
# These match utils/config.py so the whole team is consistent

DATA_DIR = Path('../data')   # adjust if your folder is named differently
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')

CLASSES = ['acne', 'hyperpigmentation', 'Nail_psoriasis', 'SJS-TEN', 'Vitiligo']
NUM_CLASSES = len(CLASSES)
IMG_SIZE = (224, 224)        # standard for ResNet50 input
BATCH_SIZE = 32
EPOCHS_BASELINE = 20        # for CNN from scratch
EPOCHS_TL = 10              # for transfer learning fine-tuning
RANDOM_SEED = 42

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Config loaded ✅')
print(f'Classes: {CLASSES}')
print(f'Image size: {IMG_SIZE}, Batch: {BATCH_SIZE}')

## 2. Data Loading & Preprocessing

We use `ImageDataGenerator` which:
- Loads images from folders (folder name = class label automatically)
- Resizes to 224×224
- Normalizes pixel values from [0, 255] → [0, 1]
- Applies augmentation to the training set to help with class imbalance

In [ ]:
# Verify dataset path and class folders exist
print('Checking dataset structure...')
for cls in CLASSES:
    path = DATA_DIR / cls
    if path.exists():
        count = len(list(path.glob('*')))
        print(f'  ✅ {cls}: {count} images')
    else:
        print(f'  ❌ {cls}: folder not found — check DATA_DIR')

In [ ]:
# Training data generator — with augmentation
# WHY augmentation? Our dataset is imbalanced (acne=1148, SJS-TEN=3164)
# Augmentation artificially increases variety so the model doesn't overfit to majority classes

train_datagen = ImageDataGenerator(
    rescale=1./255,           # normalize pixels to [0,1]
    rotation_range=20,        # randomly rotate images ±20°
    width_shift_range=0.2,    # randomly shift horizontally
    height_shift_range=0.2,   # randomly shift vertically
    shear_range=0.2,          # shear transformation
    zoom_range=0.2,           # random zoom
    horizontal_flip=True,     # flip left-right (skin lesions look same either way)
    fill_mode='nearest',      # fill empty pixels after transforms
    validation_split=0.2      # 80% train, 20% val
)

# Test/val generator — NO augmentation, just normalize
test_datagen = ImageDataGenerator(rescale=1./255)

print('Data generators created ✅')

In [ ]:
# Load training set
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',   # one-hot encoding for multi-class
    subset='training',
    seed=RANDOM_SEED
)

# Load validation set
val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=RANDOM_SEED
)

print('\nClass indices (label → number mapping):')
print(train_generator.class_indices)

In [ ]:
# Visualize a batch of training images to sanity check loading
images, labels = next(train_generator)
class_names = list(train_generator.class_indices.keys())

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(class_names[np.argmax(labels[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Images (after augmentation)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_images.png', dpi=150)
plt.show()

---
## Part A — Baseline CNN from Scratch

**Why build from scratch first?**  
A baseline CNN gives us a performance floor. If transfer learning doesn't significantly beat this, something is wrong. It also helps us understand what the model is actually learning.

### Architecture: Conv → Pool → Conv → Pool → Flatten → Dense
- **Conv2D**: learns spatial features (edges, textures, patterns)
- **MaxPooling2D**: reduces spatial size, keeps dominant features
- **BatchNormalization**: stabilizes training, speeds convergence
- **Dropout**: randomly disables neurons to prevent overfitting
- **Dense**: final classification layer

In [ ]:
def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=5):
    """
    Baseline CNN from scratch.
    3 convolutional blocks + dense head.
    """
    model = models.Sequential([
        # ── Block 1 ──────────────────────────────────────────────
        # 32 filters, each 3×3 — learns basic edges & colors
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),   # 224×224 → 112×112
        layers.Dropout(0.25),

        # ── Block 2 ──────────────────────────────────────────────
        # 64 filters — learns more complex textures
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),   # 112×112 → 56×56
        layers.Dropout(0.25),

        # ── Block 3 ──────────────────────────────────────────────
        # 128 filters — learns high-level patterns (lesion shapes)
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),   # 56×56 → 28×28
        layers.Dropout(0.25),

        # ── Classifier Head ──────────────────────────────────────
        layers.Flatten(),             # 28×28×128 → 100352 vector
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),          # higher dropout before final layer
        layers.Dense(num_classes, activation='softmax')  # 5 class probabilities
    ], name='baseline_cnn')

    return model


baseline_model = build_baseline_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
baseline_model.summary()

In [ ]:
# Compile the model
# - Adam optimizer: adaptive learning rate, works well out of the box
# - categorical_crossentropy: standard loss for multi-class classification
# - accuracy: easy to interpret metric

baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks — these run after each epoch automatically
callbacks_baseline = [
    # Stop early if val_loss doesn't improve for 5 epochs (saves time)
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    
    # Save the best model weights automatically
    ModelCheckpoint(MODELS_DIR / 'baseline_cnn_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    
    # Reduce learning rate if stuck (helps escape plateaus)
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print('Model compiled ✅')

In [ ]:
# Train baseline CNN
print('Training Baseline CNN...')
print('─' * 50)

history_baseline = baseline_model.fit(
    train_generator,
    epochs=EPOCHS_BASELINE,
    validation_data=val_generator,
    callbacks=callbacks_baseline,
    verbose=1
)

In [ ]:
def plot_training_history(history, title='Training History', save_path=None):
    """Plot accuracy and loss curves side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

plot_training_history(history_baseline, 'Baseline CNN',
                      save_path=RESULTS_DIR / 'baseline_training_curves.png')

---
## Part B — Transfer Learning with ResNet50

**Why transfer learning?**  
ResNet50 was pretrained on ImageNet (1.2M images, 1000 classes). It already knows how to detect edges, textures, and shapes. We just replace the final classification layer and fine-tune it for our 5 skin disease classes.

**Strategy (2 phases):**
1. **Freeze base** → train only our new classification head (fast, ~5 epochs)
2. **Unfreeze top layers** → fine-tune the whole network at a low learning rate

In [ ]:
def build_transfer_model(num_classes=5):
    """
    ResNet50 with custom classification head.
    Base layers frozen initially.
    """
    # Load ResNet50 pretrained on ImageNet, without its top classifier
    base_model = ResNet50(
        weights='imagenet',      # use pretrained weights
        include_top=False,       # remove ImageNet's 1000-class head
        input_shape=(*IMG_SIZE, 3)
    )

    # Freeze all base layers — we don't want to overwrite ImageNet knowledge yet
    base_model.trainable = False
    print(f'Base model layers: {len(base_model.layers)}')
    print(f'Trainable layers: {len([l for l in base_model.layers if l.trainable])}')

    # Build our custom head on top
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)  # training=False keeps BatchNorm frozen
    x = layers.GlobalAveragePooling2D()(x)  # better than Flatten for transfer learning
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='resnet50_transfer')
    return model, base_model


tl_model, base_model = build_transfer_model(NUM_CLASSES)
tl_model.summary()

In [ ]:
# Phase 1 — Train only the head (base frozen)
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_tl_phase1 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODELS_DIR / 'resnet50_phase1_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('Phase 1: Training classification head only (base frozen)...')
print('─' * 50)

history_tl_phase1 = tl_model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_tl_phase1,
    verbose=1
)

In [ ]:
# Phase 2 — Fine-tune: unfreeze top layers of ResNet50
# We unfreeze from layer 100 onwards (keeps early edge/texture detectors frozen)

base_model.trainable = True
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = len([l for l in base_model.layers if l.trainable])
print(f'Unfrozen layers for fine-tuning: {trainable_count}')

# IMPORTANT: use a much lower learning rate for fine-tuning
# Too high = destroys pretrained weights
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_tl_phase2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODELS_DIR / 'resnet50_finetuned_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print('\nPhase 2: Fine-tuning unfrozen ResNet50 layers...')
print('─' * 50)

history_tl_phase2 = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=val_generator,
    callbacks=callbacks_tl_phase2,
    verbose=1
)

In [ ]:
# Plot fine-tuning curves
plot_training_history(history_tl_phase2, 'ResNet50 Fine-tuning',
                      save_path=RESULTS_DIR / 'resnet50_finetuning_curves.png')

---
## 5. Compare Results

Compare baseline CNN vs ResNet50 on the validation set.

In [ ]:
# Evaluate both models on validation set
print('Evaluating Baseline CNN...')
baseline_loss, baseline_acc = baseline_model.evaluate(val_generator, verbose=0)

print('Evaluating ResNet50 (fine-tuned)...')
tl_loss, tl_acc = tl_model.evaluate(val_generator, verbose=0)

print('\n' + '─' * 40)
print(f'{'Model':<25} {'Val Accuracy':>12} {'Val Loss':>10}')
print('─' * 40)
print(f'{'Baseline CNN':<25} {baseline_acc:>11.4f} {baseline_loss:>10.4f}')
print(f'{'ResNet50 (fine-tuned)':<25} {tl_acc:>11.4f} {tl_loss:>10.4f}')
print('─' * 40)

In [ ]:
# Confusion matrix for the best model (ResNet50)
val_generator.reset()
y_pred = tl_model.predict(val_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_generator.classes
class_names = list(val_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — ResNet50 (Fine-tuned)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred_classes, target_names=class_names))

In [ ]:
# Save final model summary to results
results_summary = {
    'Model': ['Baseline CNN', 'ResNet50 Fine-tuned'],
    'Val Accuracy': [round(baseline_acc, 4), round(tl_acc, 4)],
    'Val Loss': [round(baseline_loss, 4), round(tl_loss, 4)]
}

results_df = pd.DataFrame(results_summary)
results_df.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
print('Results saved to results/model_comparison.csv')
results_df